# Implicit Demographic Biases in Large Language Models
## Full Research Pipeline — Adapting Fulgu & Capraro (2024)

This notebook implements the complete experimental pipeline for two studies:
- **Study 1**: Intersectional Occupational Stereotyping (Implicit Attribution)
- **Study 2**: Algorithmic Medical Triage (Moral Dilemma Bias)

### Pipeline Stages
1. **Setup** — Environment, configuration, stimulus review
2. **Data Collection** — API calls (or synthetic data for testing)
3. **Response Parsing** — Extract structured fields from raw LLM outputs
4. **Study 1 Analysis** — Inclusivity index, t-tests, chi-square, logistic regression
5. **Study 2 Analysis** — Factorial regression, refusal analysis, Tukey HSD
6. **Cross-Model Comparison** — Side-by-side results across GPT-4.1, Claude 3.5, Gemini 2.5

---

## 1. Setup & Environment

In [ ]:
# ── Core imports ──
import os, sys, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# ── Project imports ──
from config import MODELS, STUDY1_CONFIG, STUDY2_CONFIG
from stimuli import (
    ALL_STUDY1_PHRASES, HIGH_STATUS_PHRASES, SUPPORT_STATUS_PHRASES,
    CONTROL_PHRASES, STUDY2_CONDITIONS,
    get_study1_prompt, get_study2_prompt,
)
from parsers import parse_study1_csv, parse_study2_csv
from study1_analysis import (
    compute_inclusivity_index, compute_category_inclusivity,
    ttest_inclusivity, chi_square_race, logistic_regression_white_male,
    plot_inclusivity_comparison, plot_racial_distribution, plot_gender_by_phrase,
    run_full_study1_analysis,
)
from study2_analysis import (
    descriptive_by_condition, factorial_regression,
    refusal_analysis, tukey_hsd_conditions,
    plot_condition_bars, plot_interaction, plot_coefficient_forest,
    plot_refusal_rates, run_full_study2_analysis,
)

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Study 1 stimuli: {len(ALL_STUDY1_PHRASES)} phrases")
print(f"Study 2 conditions: {len(STUDY2_CONDITIONS)} patient profiles")
print(f"Models configured: {list(MODELS.keys())}")
print("\n✓ Environment ready.")

### 1.1 Review Stimuli

In [ ]:
# ── Study 1: Preview all phrases ──
print("="*80)
print("STUDY 1 STIMULUS SET")
print("="*80)

for category, phrases in [("HIGH-STATUS", HIGH_STATUS_PHRASES),
                           ("SUPPORT-STATUS", SUPPORT_STATUS_PHRASES),
                           ("CONTROL", CONTROL_PHRASES)]:
    print(f"\n--- {category} ({len(phrases)} phrases) ---")
    for p in phrases:
        print(f"  [{p.phrase_id}] ({p.industry}) {p.text[:90]}...")

In [ ]:
# ── Study 2: Preview all conditions and prompts ──
print("="*80)
print("STUDY 2 CONDITIONS (2×2×2 Factorial)")
print("="*80)

cond_data = []
for c in STUDY2_CONDITIONS:
    cond_data.append({
        "ID": c.condition_id,
        "Age": c.age,
        "Race": c.race,
        "SES": c.ses,
        "Profile": c.profile_text,
    })
pd.DataFrame(cond_data)

In [ ]:
# ── Preview exact prompts ──
print("STUDY 1 — Example prompt:")
print(get_study1_prompt(HIGH_STATUS_PHRASES[0]))
print()
print("STUDY 2 — Example prompt:")
print(get_study2_prompt(STUDY2_CONDITIONS[0]))

---
## 2. Data Collection

**Two options:**
- **Option A**: Generate synthetic data to test the pipeline (no API keys needed)
- **Option B**: Run real API collection (requires API keys set as env vars)

Start with Option A to validate the pipeline, then switch to Option B for real data.

### Option A: Synthetic Data (Pipeline Testing)

In [ ]:
from synthetic_data import generate_study1_synthetic, generate_study2_synthetic

# Generate synthetic data with realistic bias patterns baked in
generate_study1_synthetic(
    output_csv="data/study1_results.csv",
    models=["gpt-4.1", "claude-3.5-sonnet", "gemini-2.5-pro"],
    iterations=50,
)

generate_study2_synthetic(
    output_csv="data/study2_results.csv",
    models=["gpt-4.1", "claude-3.5-sonnet", "gemini-2.5-pro"],
    iterations=30,
)

### Option B: Real API Collection

**Prerequisites:**
```bash
export OPENAI_API_KEY="sk-..."
export ANTHROPIC_API_KEY="sk-ant-..."
export GOOGLE_API_KEY="AIza..."
```

Uncomment the cells below to run real collection.

In [ ]:
# ── UNCOMMENT TO RUN REAL COLLECTION ──

# from collectors import collect_study1, collect_study2

# # Pilot: 3 iterations per phrase, single model
# collect_study1(
#     model_keys=["gpt-4.1"],
#     iterations=3,
#     output_csv="data/study1_pilot.csv",
# )

# # Full collection (WARNING: ~3,750 API calls for Study 1, ~720 for Study 2)
# collect_study1(output_csv="data/study1_results.csv")
# collect_study2(output_csv="data/study2_results.csv")

In [ ]:
# ── DRY RUN: Preview what would be sent without making calls ──

# from collectors import collect_study1
# collect_study1(
#     model_keys=["gpt-4.1"],
#     phrase_ids=["high_tech_1", "support_tech_1"],
#     iterations=2,
#     dry_run=True,
# )

---
## 3. Response Parsing

Parse raw LLM text responses into structured data:
- **Study 1**: Name, Age, Gender, Race/Ethnicity (from name)
- **Study 2**: Likert score (integer 1-7), refusal detection

In [ ]:
# ── Parse Study 1 ──
s1_parsed = parse_study1_csv(
    input_csv="data/study1_results.csv",
    output_csv="data/study1_parsed.csv",
    race_method="simple",  # Change to "ethnicolr" if installed
)

print(f"\nStudy 1 parsed: {len(s1_parsed)} rows")
print(f"Gender parse rate: {s1_parsed['parsed_gender'].notna().mean()*100:.1f}%")
print(f"Name parse rate: {s1_parsed['parsed_name'].notna().mean()*100:.1f}%")
print(f"Refusal rate: {s1_parsed['is_refusal'].mean()*100:.1f}%")
s1_parsed.head(3)

In [ ]:
# ── Parse Study 2 ──
s2_parsed = parse_study2_csv(
    input_csv="data/study2_results.csv",
    output_csv="data/study2_parsed.csv",
)

print(f"\nStudy 2 parsed: {len(s2_parsed)} rows")
print(f"Likert parse rate: {s2_parsed['likert_score'].notna().mean()*100:.1f}%")
print(f"Refusal rate: {s2_parsed['is_refusal'].mean()*100:.1f}%")
s2_parsed.head(3)

---
## 4. Study 1 Analysis: Intersectional Occupational Stereotyping

Replicating Fulgu & Capraro (2024) methodology:
- Inclusivity Index per phrase and per category
- T-test: I_high vs I_support
- Racial attribution distributions
- Intersectional logistic regression

### 4.1 Inclusivity Index Computation

In [ ]:
s1_df = pd.read_csv("data/study1_parsed.csv")

# Compute per model
for model_name in s1_df['model'].unique():
    mdf = s1_df[s1_df['model'] == model_name]
    idx = compute_inclusivity_index(mdf)
    cats = compute_category_inclusivity(idx)
    
    print(f"\n{'='*60}")
    print(f"MODEL: {model_name}")
    print(f"{'='*60}")
    print(f"  I_high    = {cats['I_high']['mean']:.4f} ± {cats['I_high']['se']:.4f}  (n={cats['I_high']['n']})")
    print(f"  I_support = {cats['I_support']['mean']:.4f} ± {cats['I_support']['se']:.4f}  (n={cats['I_support']['n']})")
    print(f"\n  Per-phrase breakdown:")
    print(idx[['phrase_id', 'role_level', 'inclusivity_index', 'n_valid']].to_string(index=False))

### 4.2 Primary Hypothesis Test: T-Test

In [ ]:
for model_name in s1_df['model'].unique():
    mdf = s1_df[s1_df['model'] == model_name]
    idx = compute_inclusivity_index(mdf)
    result = ttest_inclusivity(idx)
    
    print(f"\n{'='*60}")
    print(f"T-TEST: {model_name}")
    print(f"{'='*60}")
    print(f"  {result['interpretation']}")
    print(f"  t({result['df']}) = {result['t_statistic']:.3f}")
    print(f"  p = {result['p_value']:.6f}")
    print(f"  Cohen's d = {result['cohens_d']:.3f}")

### 4.3 Inclusivity Index Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for i, model_name in enumerate(s1_df['model'].unique()):
    mdf = s1_df[s1_df['model'] == model_name]
    idx = compute_inclusivity_index(mdf)
    cats = compute_category_inclusivity(idx)
    
    labels = ["High-Status", "Support-Status"]
    means = [cats['I_high']['mean'], cats['I_support']['mean']]
    ses = [cats['I_high']['se'], cats['I_support']['se']]
    colors = ['#2E5984', '#C75B3F']
    
    bars = axes[i].bar(labels, means, yerr=ses, capsize=8, color=colors,
                       edgecolor='white', linewidth=1.5, width=0.5)
    axes[i].set_title(model_name, fontsize=13, fontweight='bold')
    axes[i].set_ylim(0, 1.0)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)
    
    for bar, m, se in zip(bars, means, ses):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + se + 0.03,
                     f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

axes[0].set_ylabel('Inclusivity Index', fontsize=12)
fig.suptitle('Inclusivity Index by Role Level (Replicating Fulgu & Capraro Fig. 1)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/study1_inclusivity_comparison_all.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Gender Attribution per Phrase

In [ ]:
for model_name in s1_df['model'].unique():
    mdf = s1_df[s1_df['model'] == model_name]
    fig = plot_gender_by_phrase(mdf, model_name,
                                save_path=f'outputs/study1_gender_phrase_{model_name}.png')
    plt.show()
    plt.close()

### 4.5 Racial Attribution Distribution

In [ ]:
for model_name in s1_df['model'].unique():
    mdf = s1_df[s1_df['model'] == model_name]
    
    # Chi-square test
    chi2_result = chi_square_race(mdf)
    print(f"\n{model_name}: {chi2_result['interpretation']}")
    
    # Plot
    fig = plot_racial_distribution(mdf, model_name,
                                   save_path=f'outputs/study1_racial_{model_name}.png')
    plt.show()
    plt.close()

### 4.6 Intersectional Logistic Regression: P(White Male)

In [ ]:
logreg = logistic_regression_white_male(s1_df)

if 'error' in logreg:
    print(f"Error: {logreg['error']}")
else:
    print("LOGISTIC REGRESSION: Predicting P(White Male Attribution)")
    print("="*60)
    print(f"N = {logreg['n_obs']}")
    print(f"Pseudo R² = {logreg['pseudo_r2']:.4f}")
    print(f"\n{logreg['interpretation']}")
    print("\nOdds Ratios:")
    for k, v in logreg['odds_ratios'].items():
        p = logreg['p_values'][k]
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f"  {k}: OR = {v:.3f}, p = {p:.4f} {sig}")

---
## 5. Study 2 Analysis: Algorithmic Medical Triage

Adapting Fulgu & Capraro (2024) moral dilemma methodology:
- Descriptive statistics by condition
- Full factorial regression (Age × Race × SES)
- Refusal rate analysis
- Post-hoc comparisons

### 5.1 Descriptive Statistics

In [ ]:
s2_df = pd.read_csv("data/study2_parsed.csv")

for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    desc = descriptive_by_condition(mdf)
    
    print(f"\n{'='*70}")
    print(f"DESCRIPTIVES: {model_name}")
    print(f"{'='*70}")
    print(desc[['condition_id', 'age', 'race', 'ses', 'mean', 'std', 'se', 'n', 'refusal_rate']].to_string(index=False))

### 5.2 Condition Bar Charts

In [ ]:
for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    desc = descriptive_by_condition(mdf)
    fig = plot_condition_bars(desc, model_name,
                              save_path=f'outputs/study2_conditions_{model_name}.png')
    plt.show()
    plt.close()

### 5.3 Factorial Linear Regression

In [ ]:
for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    reg = factorial_regression(mdf, include_model=False)
    
    print(f"\n{'='*70}")
    print(f"FACTORIAL REGRESSION: {model_name}")
    print(f"{'='*70}")
    
    if 'error' in reg:
        print(f"Error: {reg['error']}")
    else:
        print(f"R² = {reg['r_squared']:.4f}, Adj R² = {reg['adj_r_squared']:.4f}")
        print(f"F = {reg['f_statistic']:.3f}, p = {reg['f_pvalue']:.6f}")
        print(f"N = {reg['n_obs']}")
        print(f"\n{reg['interpretation']}")
        
        # Coefficient plot
        fig = plot_coefficient_forest(reg,
                                      save_path=f'outputs/study2_coefs_{model_name}.png')
        plt.show()
        plt.close()

### 5.4 Interaction Plots

In [ ]:
for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for ax, (f1, f2) in zip(axes, [('race', 'age'), ('race', 'ses'), ('age', 'ses')]):
        valid = mdf.dropna(subset=['likert_score'])
        agg = valid.groupby([f1, f2])['likert_score'].agg(['mean', 'sem']).reset_index()
        for level, grp in agg.groupby(f1):
            grp = grp.sort_values(f2)
            ax.errorbar(grp[f2].astype(str), grp['mean'], yerr=grp['sem'],
                       marker='o', capsize=5, linewidth=2, markersize=8, label=str(level))
        ax.set_xlabel(f2.upper(), fontsize=11)
        ax.set_ylabel('Mean Likert' if ax == axes[0] else '', fontsize=11)
        ax.set_title(f'{f1.capitalize()} × {f2.capitalize()}', fontsize=12, fontweight='bold')
        ax.legend(title=f1.capitalize())
        ax.set_ylim(0, 7.5)
        ax.axhline(y=4, color='gray', linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    fig.suptitle(f'Interaction Plots — {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'outputs/study2_interactions_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

### 5.5 Refusal Rate Analysis

In [ ]:
for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    ref = refusal_analysis(mdf)
    
    print(f"\n{'='*60}")
    print(f"REFUSAL ANALYSIS: {model_name}")
    print(f"{'='*60}")
    print(f"Total refusals: {ref['total_refusals']} ({ref['refusal_rate']*100:.1f}%)")
    print(f"{ref['interpretation']}")
    
    if ref['total_refusals'] > 0:
        print(f"\nBy race: {ref.get('by_race', {})}")
        print(f"By age: {ref.get('by_age', {})}")
        print(f"By SES: {ref.get('by_ses', {})}")
        
        fig = plot_refusal_rates(ref, model_name,
                                 save_path=f'outputs/study2_refusals_{model_name}.png')
        plt.show()
        plt.close()

### 5.6 Tukey HSD Post-Hoc Comparisons

In [ ]:
for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    tukey = tukey_hsd_conditions(mdf)
    
    print(f"\n{'='*60}")
    print(f"TUKEY HSD: {model_name}")
    print(f"{'='*60}")
    
    if 'error' in tukey:
        print(tukey['error'])
    else:
        print(f"Significant pairs: {tukey['n_significant']} / {tukey['n_total_comparisons']}")
        if tukey['n_significant'] > 0:
            sig = tukey['significant_pairs']
            print(sig[['group1', 'group2', 'meandiff', 'p-adj', 'reject']].to_string(index=False))

---
## 6. Cross-Model Comparison

In [ ]:
# ── Cross-model factorial regression (all data pooled, model as fixed effect) ──
cross_reg = factorial_regression(s2_df, include_model=True)

if 'error' not in cross_reg:
    print("CROSS-MODEL FACTORIAL REGRESSION")
    print("="*60)
    print(f"R² = {cross_reg['r_squared']:.4f}")
    print(f"N = {cross_reg['n_obs']}")
    print(f"\n{cross_reg['interpretation']}")
    
    fig = plot_coefficient_forest(cross_reg,
                                  save_path='outputs/study2_coefs_all_models.png')
    plt.show()
    plt.close()
else:
    print(f"Error: {cross_reg['error']}")

In [ ]:
# ── Summary comparison table ──
summary_rows = []

for model_name in s2_df['model'].unique():
    mdf = s2_df[s2_df['model'] == model_name]
    desc = descriptive_by_condition(mdf)
    reg = factorial_regression(mdf, include_model=False)
    ref = refusal_analysis(mdf)
    
    row = {'Model': model_name}
    
    # Grand mean
    row['Grand Mean'] = mdf['likert_score'].dropna().mean()
    row['Refusal Rate'] = ref['refusal_rate']
    
    # Key coefficients
    if 'error' not in reg:
        row['R²'] = reg['r_squared']
        for var in ['age_code', 'race_code', 'ses_code']:
            if var in reg['coefficients']:
                row[f'β_{var}'] = reg['coefficients'][var]
                row[f'p_{var}'] = reg['p_values'][var]
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\nCROSS-MODEL SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

---
## 7. Export Results

Save all analysis outputs for the manuscript.

In [ ]:
# ── Run full automated analysis (saves all plots and tables) ──
os.makedirs('outputs', exist_ok=True)

print("Running full Study 1 analysis...")
s1_results = run_full_study1_analysis('data/study1_parsed.csv', 'outputs')

print("\nRunning full Study 2 analysis...")
s2_results = run_full_study2_analysis('data/study2_parsed.csv', 'outputs')

print("\n" + "="*60)
print("ALL OUTPUTS SAVED TO: outputs/")
print("="*60)
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

---
## Appendix: Sample Size & Power Verification

In [ ]:
from scipy.stats import norm

def power_ttest(d, n1, n2, alpha=0.05):
    """Compute power for independent t-test given Cohen's d and sample sizes."""
    se = np.sqrt(1/n1 + 1/n2)
    z_crit = norm.ppf(1 - alpha/2)
    ncp = d / se  # non-centrality parameter
    power = 1 - norm.cdf(z_crit - ncp) + norm.cdf(-z_crit - ncp)
    return power

print("POWER ANALYSIS")
print("="*50)
print("\nStudy 1 (Inclusivity Index t-test):")
for d in [0.5, 1.0, 2.0]:
    pwr = power_ttest(d, n1=10, n2=10, alpha=0.05)
    print(f"  d = {d:.1f}, n_per_group = 10 phrases: power = {pwr:.3f}")

print("\nStudy 2 (per-cell, single model):")
for d in [0.3, 0.5, 0.8]:
    pwr = power_ttest(d, n1=30, n2=30, alpha=0.05)
    print(f"  d = {d:.1f}, n = 30 per cell: power = {pwr:.3f}")

print("\nFulgu & Capraro reported d > 2.0 for primary comparisons.")
print("Our design is powered for medium effects (d ≥ 0.5) at > 0.85.")